# Libraires and Set Up

In [1]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

import spacy
nlp = spacy.load("en_core_web_sm")

from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

c:\Users\UCX37\AppData\Local\miniconda3\envs\intent-bert\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\UCX37\AppData\Local\miniconda3\envs\intent-bert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the data

In [2]:
df = pd.read_csv(r"C:\work_project_1\Output\conversation_flow_dataset.csv")

print(df.shape)
df.head()

(5459, 5)


,conversation_id,source_file,full_conversation,caller_text,num_turns
0,DC111969W0000018609W_20260310063145,C:\work_project_1\UNKNOWN_SKILL\UNKNOWN_SKILL\...,agent: Jota and thank you for calling J.B. Hi....,NaN,8
1,DC183307W0000018898W_20260311010733,C:\work_project_1\UNKNOWN_SKILL\UNKNOWN_SKILL\...,"agent: Welcome to Harvey Norman, I was speakin...","Hi, Mr. Havila from Prada Kian, I speak to any...",77
2,DC184879W0000018916W_20260311012223,C:\work_project_1\UNKNOWN_SKILL\UNKNOWN_SKILL\...,agent: Thank you for calling the Harvey Norman...,Brayheys Josh how you doing? Good. I've got a ...,35
3,DC261734W0000002181W_20260205052219,C:\work_project_1\UNKNOWN_SKILL\UNKNOWN_SKILL\...,agent: Thank you for calling the Harvey Norman...,"Lauren, this is Calvinia? Good, good. I've got...",47
4,DC278954W0000007802W_20260217002204,C:\work_project_1\UNKNOWN_SKILL\UNKNOWN_SKILL\...,agent: Welcome to Harvey Norman Achuka.\ncalle...,"Good morning, right? This is Kelvin from one o...",38


In [3]:
df["caller_text"] = df["caller_text"].fillna("").astype(str)

# Removing Custom Stopwords

In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = set(ENGLISH_STOP_WORDS)

call_center_noise = {
    "yeah","yep","ok","okay","right","oh","um","uh",
    "thank","thanks","bye","goodbye","please","just",
    "im","ive","ill","youre","dont","cant",
    "press","number","message","option","menu",
    "yes","hello","hi","good","sorry","leave"
}

domain_noise = {
    "harvey","norman","team","agent","speaking",
    "calling","call","phone","email",
    "help","customer","service"
}

custom_stopwords = custom_stopwords.union(call_center_noise, domain_noise)

# Cleaning and Lemmatization of Text

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()

    doc = nlp(text)

    words = [
        token.lemma_
        for token in doc
        if token.lemma_ not in custom_stopwords
        and token.is_alpha
        and len(token.lemma_) > 2
    ]

    return " ".join(words)

df["clean_text"] = df["caller_text"].apply(clean_text)

df[["caller_text", "clean_text"]].head()

,caller_text,clean_text
0,,
1,"Hi, Mr. Havila from Prada Kian, I speak to any...",havila prada kian speak computer department je...
2,Brayheys Josh how you doing? Good. I've got a ...,brayhey josh want change stock credit burley w...
3,"Lauren, this is Calvinia? Good, good. I've got...",lauren calvinia line moment old cold case say ...
4,"Good morning, right? This is Kelvin from one o...",morning kelvin follow technical computer photo...


# Cleaning the full conversation

In [6]:
# APPLY CLEANING TO FULL CONVERSATIONS

print("STEP 1: Cleaning full_conversation column")

# Apply the SAME clean_text function to full_conversation
df['clean_conversation'] = df['full_conversation'].apply(clean_text)

# Check the results
print("\n✓ Cleaning complete!")
print(f"  Original conversations: {len(df):,}")
print(f"  After cleaning (non-empty): {len(df[df['clean_conversation'] != '']):,}")

# Show example of cleaning
print("\n--- EXAMPLE OF CLEANING ---")
print(f"ORIGINAL: {df['full_conversation'].iloc[0][:200]}...")
print(f"\nCLEANED:  {df['clean_conversation'].iloc[0][:200]}...")

# Remove empty conversations
df_clean = df[df['clean_conversation'] != ''].copy()

# OPTIONAL: Remove very short conversations (adjust number as needed)
min_words = 20
df_clean = df_clean[df_clean['clean_conversation'].str.split().str.len() >= min_words]
print(f"\n✓ After removing conversations with <{min_words} words: {len(df_clean):,} conversations")

# Check conversation length statistics
lengths = df_clean['clean_conversation'].str.split().str.len()
print(f"\n--- CONVERSATION LENGTH STATS (words after cleaning) ---")
print(f"  Minimum: {lengths.min():.0f} words")
print(f"  Maximum: {lengths.max():.0f} words")
print(f"  Average: {lengths.mean():.1f} words")
print(f"  Median: {lengths.median():.0f} words")

STEP 1: Cleaning full_conversation column

✓ Cleaning complete!
  Original conversations: 5,459
  After cleaning (non-empty): 5,431

--- EXAMPLE OF CLEANING ---
ORIGINAL: agent: Jota and thank you for calling J.B. Hi. Hi. Hi. Support Office. Please select from the following options.
agent: Press 1 for online customer service. Press 2. For insurance. Press 3 for commerc...

CLEANED:  jota support office select follow online insurance commercial wait transfer jota jaby hifi insurance outside operating hour monday friday hour like send send claim jbhifyco...

✓ After removing conversations with <20 words: 4,314 conversations

--- CONVERSATION LENGTH STATS (words after cleaning) ---
  Minimum: 20 words
  Maximum: 1560 words
  Average: 149.1 words
  Median: 114 words


# Segmeting conversations into utterances 
# Clustering Using Segements


In [9]:
# SEGMENT CONVERSATIONS INTO UTTERANCES

print("SEGMENTING CONVERSATIONS INTO UTTERANCES")

import re

def segment_conversation(text):
    """
    Split conversation into individual utterances/sentences
    """
    if not isinstance(text, str):
        return []
    
    # Split on common conversation markers
    segments = []
    
    # Split by periods, question marks, exclamation marks
    sentences = re.split(r'[.!?]+', text)
    
    # Filter out empty segments
    sentences = [s.strip().lower() for s in sentences if len(s.strip()) > 15]
    
    return sentences

def get_first_n_segments(text, n=10):
    """
    Get first N meaningful segments of a conversation
    """
    segments = segment_conversation(text)
    return segments[:n]

# Apply segmentation to your cleaned conversations
print("\nSegmenting conversations...")
df_clean['segments'] = df_clean['clean_conversation'].apply(segment_conversation)
df_clean['first_10_segments'] = df_clean['clean_conversation'].apply(lambda x: get_first_n_segments(x, 10))

# Check segment distribution
segment_lengths = df_clean['segments'].apply(len)
print(f"Average segments per conversation: {segment_lengths.mean():.1f}")
print(f"Conversations with >5 segments: {(segment_lengths > 5).sum():,}")
print(f"Conversations with >10 segments: {(segment_lengths > 10).sum():,}")

# Show example
print("\nExample of segmented conversation:")
sample_conv = df_clean['clean_conversation'].iloc[0]
print(f"Original: {sample_conv[:200]}...")
print(f"\nSegments (first 5):")
for i, seg in enumerate(df_clean['first_10_segments'].iloc[0][:5], 1):
    print(f"  {i}. {seg}")


# REMOVE NO_INTENT FROM FIRST SEGMENTS

print("\n" + "=" * 60)
print("REMOVING NO_INTENT FROM FIRST SEGMENTS")
print("=" * 60)

# Define keywords that indicate NO_INTENT (greetings, hold, IVR)
no_intent_keywords = [
    'welcome', 'hello', 'hi', 'good morning', 'good afternoon',
    'thank you for calling', 'please hold', 'hold the line',
    'press', 'enter', 'option', 'menu', 'ivr', 'automated',
    'call recorded', 'quality assurance', 'voice mail', 'voicemail',
    'please listen', 'carefully', 'currently unavailable'
]

def has_intent_in_first_segments(segments, min_length=5):
    """
    Check if the first meaningful segments contain actual intent
    """
    if not segments:
        return False, [], "No segments"
    
    # Check first 5 segments (or all if less)
    check_segments = segments[:5]
    
    # Filter out segments that are just no_intent
    meaningful_segments = []
    for seg in check_segments:
        # Skip if segment is too short
        if len(seg.split()) < min_length:
            continue
        
        # Skip if segment contains no_intent keywords
        is_no_intent = any(keyword in seg for keyword in no_intent_keywords)
        
        if not is_no_intent:
            meaningful_segments.append(seg)
    
    # If we have meaningful segments, there is intent
    has_intent = len(meaningful_segments) > 0
    
    return has_intent, meaningful_segments, f"Found {len(meaningful_segments)} meaningful segments"

# Apply to all conversations
df_clean['has_intent'], df_clean['meaningful_segments'], df_clean['intent_check'] = zip(
    *df_clean['first_10_segments'].apply(has_intent_in_first_segments)
)

# Filter to conversations with real intent
df_with_intent = df_clean[df_clean['has_intent'] == True].copy()
df_no_intent = df_clean[df_clean['has_intent'] == False].copy()

print(f"\nResults:")
print(f"  Conversations with REAL intent: {len(df_with_intent):,} ({len(df_with_intent)/len(df_clean)*100:.1f}%)")
print(f"  Conversations with ONLY no_intent: {len(df_no_intent):,} ({len(df_no_intent)/len(df_clean)*100:.1f}%)")

# RECLUSTER USING ONLY MEANINGFUL SEGMENTS

print("RECLUSTERING USING ONLY MEANINGFUL SEGMENTS")

if len(df_with_intent) > 100:
    # Combine meaningful segments for each conversation
    df_with_intent['meaningful_text'] = df_with_intent['meaningful_segments'].apply(lambda x: ' '.join(x))
    
    # Generate embeddings for meaningful text only
    from sentence_transformers import SentenceTransformer
    
    print("\nGenerating embeddings for meaningful segments...")
    model = SentenceTransformer('all-mpnet-base-v2')
    
    meaningful_texts = df_with_intent['meaningful_text'].tolist()
    
    embeddings_meaningful = model.encode(
        meaningful_texts,
        batch_size=16,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False
    )
    
    # Reduce dimensionality with UMAP
    from umap import UMAP
    
    print("\nReducing dimensions...")
    umap_meaningful = UMAP(
        n_components=50,
        n_neighbors=10,
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )
    
    embeddings_meaningful_reduced = umap_meaningful.fit_transform(embeddings_meaningful)
    
    # Cluster with Agglomerative Clustering
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics import silhouette_score
    
    # Try different cluster counts
    best_sil = -1
    best_labels = None
    best_n = 5
    
    for n_clust in [5, 6, 7, 8, 9, 10]:
        clusterer = AgglomerativeClustering(n_clusters=n_clust, metric='euclidean', linkage='ward')
        labels = clusterer.fit_predict(embeddings_meaningful_reduced)
        
        if len(set(labels)) > 1:
            sil_score = silhouette_score(embeddings_meaningful_reduced, labels, metric='euclidean')
            print(f"  n_clusters={n_clust}: silhouette={sil_score:.3f}")
            
            if sil_score > best_sil:
                best_sil = sil_score
                best_labels = labels
                best_n = n_clust
    
    # Add clusters to dataframe
    df_with_intent['cluster'] = best_labels
    
    print(f"\nSelected {best_n} clusters with silhouette score: {best_sil:.3f}")
    

    # ANALYZE NEW CLUSTERS

    print("NEW CLUSTER ANALYSIS (No Intent Removed)")
    
    from collections import Counter
    
    print("\nCluster sizes:")
    cluster_sizes = Counter(best_labels)
    for cluster_id, size in sorted(cluster_sizes.items(), key=lambda x: x[1], reverse=True):
        percentage = (size / len(df_with_intent)) * 100
        print(f"  Cluster {cluster_id}: {size} conversations ({percentage:.1f}%)")
    
    # Show top words per cluster
    print("TOP WORDS PER CLUSTER (Meaningful Segments Only)")
    
    for cluster_id in sorted(df_with_intent['cluster'].unique()):
        cluster_data = df_with_intent[df_with_intent['cluster'] == cluster_id]
        cluster_words = ' '.join(cluster_data['meaningful_text']).split()
        cluster_counter = Counter(cluster_words)
        
        print(f"\nCLUSTER {cluster_id}:")
        for word, count in cluster_counter.most_common(15):
            print(f"  {word}: {count}")
    
    # MAP TO MANUAL INTENT LABELS
    
    print("MAPPING TO MANUAL INTENT LABELS")
    
    # Your manual intent keywords (without No_intent)
    intent_keywords_filtered = {
        'Claim_update': ['update claim', 'claim status', 'progress claim', 'follow up claim'],
        'Lodge_claim': ['lodge claim', 'new claim', 'submit claim', 'make claim', 'claim application'],
        'Claim_warranty_repair': ['warranty repair', 'repair under warranty', 'fix warranty', 'claim repair'],
        'Claim_enquiry': ['claim enquiry', 'about claim', 'claim information', 'claim question'],
        'Insurance_enquiry': ['insurance', 'insurance cover', 'policy', 'premium'],
        'Service_enquiry': ['service', 'servicing', 'service center', 'repair service'],
        'personal_information_update': ['update address', 'change number', 'update details', 'change address'],
        'invoice_enquiry': ['invoice', 'bill', 'receipt', 'payment proof'],
        'repair_delay': ['repair delay', 'delay claim', 'taking long', 'waiting repair'],
        'credit_enquiry': ['credit', 'credit note', 'store credit'],
        'billing_enquiry': ['billing', 'charge', 'statement', 'overcharge'],
        'joborder_enquiry': ['job order', 'work order', 'job number'],
        'warranty_cancellation': ['cancel warranty', 'stop warranty', 'remove warranty'],
        'Claim_follow_up': ['follow up', 'previous claim', 'claim follow', 'update on claim'],
        'complaint': ['complaint', 'unhappy', 'dissatisfied', 'bad service', 'terrible'],
        'Refund': ['refund', 'money back', 'reimbursement', 'refund claim'],
        'Cancelation': ['cancel', 'cancellation', 'stop order', 'discontinue']
    }
    
    for cluster_id in sorted(df_with_intent['cluster'].unique()):
        cluster_data = df_with_intent[df_with_intent['cluster'] == cluster_id]
        cluster_text = ' '.join(cluster_data['meaningful_text']).lower()
        
        # Score each intent
        scores = {}
        for intent, keywords in intent_keywords_filtered.items():
            score = sum(cluster_text.count(keyword) for keyword in keywords)
            scores[intent] = score
        
        # Get top intent
        best_intent = max(scores, key=scores.get)
        best_score = scores[best_intent]
        
        size = len(cluster_data)
        percentage = (size / len(df_with_intent)) * 100
        
        print(f"\nCluster {cluster_id} ({size} convos, {percentage:.1f}%):")
        print(f"  INTENT: {best_intent.replace('_', ' ').title()}")
        print(f"  Score: {best_score}")
        
        # Show sample meaningful segments
        samples = cluster_data['meaningful_segments'].head(2).tolist()
        print(f"  Examples:")
        for sample in samples:
            if sample:
                print(f"    - {sample[0][:100]}...")
    
    # Save the filtered dataset
    df_with_intent.to_csv('conversations_with_intent.csv', index=False)
    print("\n✓ Saved conversations with intent to 'conversations_with_intent.csv'")
    
else:
    print(f"\nNot enough conversations with intent ({len(df_with_intent)}) for meaningful clustering")

SEGMENTING CONVERSATIONS INTO UTTERANCES

Segmenting conversations...
Average segments per conversation: 1.0
Conversations with >5 segments: 0
Conversations with >10 segments: 0

Example of segmented conversation:
Original: jota support office select follow online insurance commercial wait transfer jota jaby hifi insurance outside operating hour monday friday hour like send send claim jbhifyco...

Segments (first 5):
  1. jota support office select follow online insurance commercial wait transfer jota jaby hifi insurance outside operating hour monday friday hour like send send claim jbhifyco

REMOVING NO_INTENT FROM FIRST SEGMENTS

Results:
  Conversations with REAL intent: 829 (19.2%)
  Conversations with ONLY no_intent: 3,485 (80.8%)
RECLUSTERING USING ONLY MEANINGFUL SEGMENTS

Generating embeddings for meaningful segments...


Batches: 100%|██████████| 52/52 [06:01<00:00,  6.96s/it]
c:\Users\UCX37\AppData\Local\miniconda3\envs\intent-bert\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Reducing dimensions...
  n_clusters=5: silhouette=0.269
  n_clusters=6: silhouette=0.292
  n_clusters=7: silhouette=0.277
  n_clusters=8: silhouette=0.266
  n_clusters=9: silhouette=0.282
  n_clusters=10: silhouette=0.277

Selected 6 clusters with silhouette score: 0.292
NEW CLUSTER ANALYSIS (No Intent Removed)

Cluster sizes:
  Cluster 2: 200 conversations (24.1%)
  Cluster 4: 184 conversations (22.2%)
  Cluster 1: 174 conversations (21.0%)
  Cluster 0: 129 conversations (15.6%)
  Cluster 5: 102 conversations (12.3%)
  Cluster 3: 40 conversations (4.8%)
TOP WORDS PER CLUSTER (Meaningful Segments Only)

CLUSTER 0:
  caller: 2433
  claim: 227
  product: 213
  speak: 169
  like: 146
  send: 141
  care: 132
  warranty: 131
  purchase: 115
  store: 106
  need: 105
  want: 102
  know: 95
  day: 90
  problem: 87

CLUSTER 1:
  caller: 866
  claim: 162
  warranty: 117
  available: 92
  speak: 87
  extend: 82
  record: 80
  group: 72
  send: 67
  product: 65
  care: 63
  person: 57
  assist: 5

# Labeling intents on full CLEANED dataset

In [10]:
# CAPTURE ALL INTENTS FROM FULL DATASET

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re

print("CAPTURING ALL INTENTS FROM FULL DATASET")

output_folder = r"C:\work_project_1\Output"
os.makedirs(output_folder, exist_ok=True)

# Your complete list of manual intents
all_intents = [
    'Claim_update', 'Lodge_claim', 'Claim_warranty_repair', 'Claim_enquiry',
    'Insurance_enquiry', 'Service_enquiry', 'personal_information_update',
    'invoice_enquiry', 'warranty_claim', 'repair_delay', 'claim_repair',
    'credit_enquiry', 'billing_enquiry', 'joborder_enquiry', 'warranty_cancellation',
    'Claim_follow_up', 'complaint', 'Refund', 'Cancelation'
]

# Define keywords for each intent (for automatic labeling)
intent_keywords = {
    'Claim_update': ['update claim', 'claim status', 'progress', 'follow up claim', 'claim update'],
    'Lodge_claim': ['lodge claim', 'new claim', 'submit claim', 'make claim', 'claim application'],
    'Claim_warranty_repair': ['warranty repair', 'repair under warranty', 'claim repair', 'repair claim'],
    'Claim_enquiry': ['claim enquiry', 'about claim', 'claim information', 'claim question'],
    'Insurance_enquiry': ['insurance', 'insurance cover', 'policy', 'premium', 'insurer'],
    'Service_enquiry': ['service', 'servicing', 'service center', 'repair service', 'service enquiry'],
    'personal_information_update': ['update address', 'change number', 'update details', 'change address', 'personal info'],
    'invoice_enquiry': ['invoice', 'bill', 'receipt', 'payment proof', 'invoice copy'],
    'warranty_claim': ['warranty claim', 'claim warranty', 'warranty', 'under warranty'],
    'repair_delay': ['repair delay', 'delay claim', 'taking long', 'waiting repair', 'delayed'],
    'claim_repair': ['claim repair', 'repair claim', 'claim and repair'],
    'credit_enquiry': ['credit', 'credit note', 'store credit', 'credit amount'],
    'billing_enquiry': ['billing', 'charge', 'statement', 'overcharge', 'bill'],
    'joborder_enquiry': ['job order', 'work order', 'job number', 'order status'],
    'warranty_cancellation': ['cancel warranty', 'stop warranty', 'remove warranty', 'warranty cancel'],
    'Claim_follow_up': ['follow up', 'previous claim', 'claim follow', 'update on claim', 'followup'],
    'complaint': ['complaint', 'unhappy', 'dissatisfied', 'bad service', 'terrible', 'poor'],
    'Refund': ['refund', 'money back', 'reimbursement', 'refund claim'],
    'Cancelation': ['cancel', 'cancellation', 'stop order', 'discontinue', 'cancel claim']
}

# Function to extract intent from conversation
def extract_intent_from_text(text):
    """Extract intent based on keyword matching"""
    if not isinstance(text, str):
        return 'Unknown'
    
    text_lower = text.lower()
    
    # Score each intent
    scores = {}
    for intent, keywords in intent_keywords.items():
        score = 0
        for keyword in keywords:
            score += text_lower.count(keyword)
        scores[intent] = score
    
    # Get highest scoring intent
    best_intent = max(scores, key=scores.get)
    best_score = scores[best_intent]
    
    # Only return if score is meaningful (at least 1 keyword match)
    if best_score >= 1:
        return best_intent
    else:
        return 'General_Enquiry'

# Use the FULL dataset (not just filtered)
print(f"\nProcessing full dataset with {len(df_clean):,} conversations")

# Apply intent extraction to full dataset
df_clean['extracted_intent'] = df_clean['clean_conversation'].apply(extract_intent_from_text)

# Count intent distribution
print("\nIntent distribution in full dataset:")
intent_counts = df_clean['extracted_intent'].value_counts()
for intent, count in intent_counts.head(15).items():
    print(f"  {intent}: {count:,} ({count/len(df_clean)*100:.1f}%)")


# CREATE LABELED DATASET FROM FULL DATA

print("CREATING LABELED DATASET FROM FULL DATA")

# Prepare the final dataset with all conversations
final_dataset_full = pd.DataFrame()

# Use the cleaned conversation text
final_dataset_full['text'] = df_clean['clean_conversation']

# Use the extracted intent
final_dataset_full['intent'] = df_clean['extracted_intent']

# Add conversation ID if available
if 'conversation_id' in df_clean.columns:
    final_dataset_full['conversation_id'] = df_clean['conversation_id']

# Remove Unknown intents (optional - keep or remove based on your needs)
final_dataset_full = final_dataset_full[final_dataset_full['intent'] != 'General_Enquiry']

# Remove empty texts
final_dataset_full = final_dataset_full[final_dataset_full['text'].str.len() > 20]

print(f"\nFinal dataset size: {len(final_dataset_full):,} conversations")

print("\nIntent distribution:")
for intent, count in final_dataset_full['intent'].value_counts().items():
    print(f"  {intent}: {count:,} ({count/len(final_dataset_full)*100:.1f}%)")

# SPLIT DATA (70-20-10)

print("SPLITTING DATA (70-20-10 RULE)")

# Split the data
X = final_dataset_full['text']
y = final_dataset_full['intent']

# First split: separate test set (10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# Second split: separate validation set (20% of remaining = 20% of original)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2222,
    random_state=42,
    stratify=y_temp
)

# Create dataframes
train_df = pd.DataFrame({'text': X_train, 'intent': y_train})
val_df = pd.DataFrame({'text': X_val, 'intent': y_val})
test_df = pd.DataFrame({'text': X_test, 'intent': y_test})

# Save to CSV
train_df.to_csv(os.path.join(output_folder, 'train_data_full.csv'), index=False)
val_df.to_csv(os.path.join(output_folder, 'validation_data_full.csv'), index=False)
test_df.to_csv(os.path.join(output_folder, 'test_data_full.csv'), index=False)
final_dataset_full.to_csv(os.path.join(output_folder, 'full_labeled_dataset_full.csv'), index=False)

print(f"\n✓ Data saved to: {output_folder}")
print("\nFile sizes:")
print(f"  train_data_full.csv: {len(train_df):,} samples ({len(train_df)/len(final_dataset_full)*100:.1f}%)")
print(f"  validation_data_full.csv: {len(val_df):,} samples ({len(val_df)/len(final_dataset_full)*100:.1f}%)")
print(f"  test_data_full.csv: {len(test_df):,} samples ({len(test_df)/len(final_dataset_full)*100:.1f}%)")
print(f"  full_labeled_dataset_full.csv: {len(final_dataset_full):,} samples (100%)")

# Show class distribution
print("CLASS DISTRIBUTION IN EACH SPLIT")

print("\nTraining set:")
for intent, count in train_df['intent'].value_counts().head(15).items():
    print(f"  {intent}: {count}")

print("\nValidation set:")
for intent, count in val_df['intent'].value_counts().head(15).items():
    print(f"  {intent}: {count}")

print("\nTest set:")
for intent, count in test_df['intent'].value_counts().head(15).items():
    print(f"  {intent}: {count}")


# SAVE INTENT MAPPING

unique_intents = final_dataset_full['intent'].unique()
intent_mapping = pd.DataFrame({
    'intent_code': range(len(unique_intents)),
    'intent_name': sorted(unique_intents)
})
intent_mapping.to_csv(os.path.join(output_folder, 'intent_mapping_full.csv'), index=False)

print(f"\n✓ Intent mapping saved ({len(unique_intents)} intents)")

# CREATE SAMPLES FOR EACH INTENT

print("SAMPLE TEXTS FOR EACH INTENT")

for intent in sorted(unique_intents):
    samples = final_dataset_full[final_dataset_full['intent'] == intent]['text'].head(3).tolist()
    print(f"\n{intent}:")
    for sample in samples[:2]:
        print(f"  - {sample[:100]}...")

print("✓ DATASET PREPARATION COMPLETE")
print(f"\nNow you have {len(final_dataset_full):,} labeled conversations with {len(unique_intents)} intent types")
print(f"All files saved to: {output_folder}")

CAPTURING ALL INTENTS FROM FULL DATASET

Processing full dataset with 4,314 conversations

Intent distribution in full dataset:
  General_Enquiry: 1,123 (26.0%)
  warranty_claim: 748 (17.3%)
  credit_enquiry: 559 (13.0%)
  invoice_enquiry: 525 (12.2%)
  Insurance_enquiry: 523 (12.1%)
  Lodge_claim: 317 (7.3%)
  Cancelation: 157 (3.6%)
  billing_enquiry: 94 (2.2%)
  Claim_update: 92 (2.1%)
  Claim_warranty_repair: 38 (0.9%)
  Claim_follow_up: 33 (0.8%)
  complaint: 25 (0.6%)
  Service_enquiry: 23 (0.5%)
  Refund: 17 (0.4%)
  personal_information_update: 16 (0.4%)
CREATING LABELED DATASET FROM FULL DATA

Final dataset size: 3,191 conversations

Intent distribution:
  warranty_claim: 748 (23.4%)
  credit_enquiry: 559 (17.5%)
  invoice_enquiry: 525 (16.5%)
  Insurance_enquiry: 523 (16.4%)
  Lodge_claim: 317 (9.9%)
  Cancelation: 157 (4.9%)
  billing_enquiry: 94 (2.9%)
  Claim_update: 92 (2.9%)
  Claim_warranty_repair: 38 (1.2%)
  Claim_follow_up: 33 (1.0%)
  complaint: 25 (0.8%)
  Service_

# Training the model

In [11]:
# TRAIN INTENT CLASSIFICATION MODEL

import os
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

print("TRAINING INTENT CLASSIFICATION MODEL")

output_folder = r"C:\work_project_1\Output"

# Load the training and validation data
train_df = pd.read_csv(os.path.join(output_folder, 'train_data_full.csv'))
val_df = pd.read_csv(os.path.join(output_folder, 'validation_data_full.csv'))
test_df = pd.read_csv(os.path.join(output_folder, 'test_data_full.csv'))

print(f"\nData loaded:")
print(f"  Training: {len(train_df):,} samples")
print(f"  Validation: {len(val_df):,} samples")
print(f"  Test: {len(test_df):,} samples")

# Clean text function
def clean_text_for_model(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
train_df['clean_text'] = train_df['text'].apply(clean_text_for_model)
val_df['clean_text'] = val_df['text'].apply(clean_text_for_model)
test_df['clean_text'] = test_df['text'].apply(clean_text_for_model)

# Remove empty texts
train_df = train_df[train_df['clean_text'].str.len() > 10]
val_df = val_df[val_df['clean_text'].str.len() > 10]
test_df = test_df[test_df['clean_text'].str.len() > 10]

print(f"\nAfter cleaning:")
print(f"  Training: {len(train_df):,} samples")
print(f"  Validation: {len(val_df):,} samples")
print(f"  Test: {len(test_df):,} samples")

# Create TF-IDF Vectorizer
print("CREATING TF-IDF FEATURES")

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df['clean_text'])
X_val = vectorizer.transform(val_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

y_train = train_df['intent']
y_val = val_df['intent']
y_test = test_df['intent']

print(f"Feature matrix: {X_train.shape}")

# Train Logistic Regression with Grid Search
print("TRAINING LOGISTIC REGRESSION MODEL")
param_grid = {
    'C': [0.1, 0.5, 1.0, 2.0],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000]
}

log_reg = LogisticRegression(class_weight='balanced', random_state=42, multi_class='ovr')

grid_search = GridSearchCV(
    log_reg, 
    param_grid, 
    cv=3, 
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("\nPerforming grid search...")
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")

# Save model and vectorizer
joblib.dump(best_model, os.path.join(output_folder, 'intent_model.pkl'))
joblib.dump(vectorizer, os.path.join(output_folder, 'vectorizer.pkl'))

print(f"\nModel saved to: {output_folder}\\intent_model.pkl")
print(f"Vectorizer saved to: {output_folder}\\vectorizer.pkl")

# Create tracker class for later use
class ConversationIntentTracker:
    def __init__(self, model, vectorizer, confidence_threshold=0.3):
        self.model = model
        self.vectorizer = vectorizer
        self.confidence_threshold = confidence_threshold
        self.intent_history = []
        self.current_intent = None
        self.intent_confidence = 0
        self.segment_count = 0
        
    def reset(self):
        self.intent_history = []
        self.current_intent = None
        self.intent_confidence = 0
        self.segment_count = 0
        
    def process_segment(self, segment_text):
        if not segment_text or len(segment_text.strip()) < 5:
            return None, 0, False
        
        cleaned = clean_text_for_model(segment_text)
        
        if len(cleaned.split()) < 2:
            return None, 0, False
        
        vec = self.vectorizer.transform([cleaned])
        pred = self.model.predict(vec)[0]
        proba = self.model.predict_proba(vec)
        confidence = max(proba[0])
        
        if confidence < self.confidence_threshold:
            return None, confidence, False
        
        changed = False
        if self.current_intent != pred:
            changed = True
            self.current_intent = pred
            self.intent_confidence = confidence
        
        self.intent_history.append({
            'segment': self.segment_count,
            'text': segment_text[:100],
            'intent': pred,
            'confidence': confidence,
            'changed': changed
        })
        
        self.segment_count += 1
        return pred, confidence, changed
    
    def get_current_intent(self):
        return self.current_intent
    
    def get_intent_flow(self):
        flow = []
        for item in self.intent_history:
            if item['changed'] or len(flow) == 0:
                flow.append({
                    'segment': item['segment'],
                    'intent': item['intent'],
                    'confidence': item['confidence']
                })
        return flow
    
    def get_summary(self):
        flow = self.get_intent_flow()
        if not flow:
            return "No intents detected"
        summary = f"Intent flow: {flow[0]['intent']}"
        for i in range(1, len(flow)):
            summary += f" -> {flow[i]['intent']}"
        return summary

print("\nTraining complete. Model ready for evaluation and dashboard.")

TRAINING INTENT CLASSIFICATION MODEL

Data loaded:
  Training: 2,233 samples
  Validation: 638 samples
  Test: 320 samples

After cleaning:
  Training: 2,233 samples
  Validation: 638 samples
  Test: 320 samples
CREATING TF-IDF FEATURES
Feature matrix: (2233, 5000)
TRAINING LOGISTIC REGRESSION MODEL

Performing grid search...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best parameters: {'C': 2.0, 'max_iter': 1000, 'solver': 'liblinear'}
Best cross-validation score: 0.424

Model saved to: C:\work_project_1\Output\intent_model.pkl
Vectorizer saved to: C:\work_project_1\Output\vectorizer.pkl

Training complete. Model ready for evaluation and dashboard.


# Model Evaluation

In [ ]:
#  MODEL EVALUATION

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns

print("MODEL EVALUATION")

output_folder = r"C:\work_project_1\Output"

# Load model and data
model = joblib.load(os.path.join(output_folder, 'intent_model.pkl'))
vectorizer = joblib.load(os.path.join(output_folder, 'vectorizer.pkl'))

test_df = pd.read_csv(os.path.join(output_folder, 'test_data_full.csv'))

# Clean function
def clean_text_for_model(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

test_df['clean_text'] = test_df['text'].apply(clean_text_for_model)
test_df = test_df[test_df['clean_text'].str.len() > 10]

X_test = vectorizer.transform(test_df['clean_text'])
y_test = test_df['intent']

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Set Accuracy: {accuracy:.3f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
print("\nConfusion Matrix (top 10 intents):")
unique_intents = y_test.unique()
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=unique_intents, columns=unique_intents)

# Show top 10 intents by frequency
top_intents = y_test.value_counts().head(10).index
cm_top = cm_df.loc[top_intents, top_intents]

print("\nConfusion Matrix for Top 10 Intents:")
print(cm_top)

# Per-class accuracy
print("\nPer-Class Accuracy:")
for intent in top_intents:
    idx = np.where(y_test == intent)[0]
    if len(idx) > 0:
        class_accuracy = (y_pred[idx] == intent).sum() / len(idx)
        print(f"  {intent}: {class_accuracy:.3f} ({len(idx)} samples)")

# Feature importance (top words per intent)
print("TOP WORDS PER INTENT")

feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_

for i, intent in enumerate(model.classes_[:10]):  # Top 10 intents
    top_features_idx = coefficients[i].argsort()[-10:][::-1]
    top_words = [feature_names[idx] for idx in top_features_idx if coefficients[i][idx] > 0]
    if top_words:
        print(f"\n{intent}:")
        print(f"  {', '.join(top_words[:8])}")

print("\nEvaluation complete. Ready for dashboard.")

MODEL EVALUATION

Test Set Accuracy: 0.762

Classification Report:
                             precision    recall  f1-score   support

                Cancelation       0.75      0.75      0.75        16
              Claim_enquiry       1.00      1.00      1.00         1
            Claim_follow_up       0.00      0.00      0.00         3
               Claim_update       0.38      0.56      0.45         9
      Claim_warranty_repair       0.20      0.25      0.22         4
          Insurance_enquiry       0.81      0.67      0.74        52
                Lodge_claim       0.71      0.78      0.75        32
                     Refund       1.00      1.00      1.00         2
            Service_enquiry       0.00      0.00      0.00         2
            billing_enquiry       0.73      0.89      0.80         9
                  complaint       0.00      0.00      0.00         3
             credit_enquiry       0.89      0.89      0.89        56
            invoice_enquiry       0

In [12]:
# COMPARE DIFFERENT ML MODELS ON BERT EMBEDDINGS

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("COMPARING DIFFERENT ML MODELS ON BERT EMBEDDINGS")

output_folder = r"C:\work_project_1\Output"

# Load the data
train_df = pd.read_csv(os.path.join(output_folder, 'train_data_full.csv'))
val_df = pd.read_csv(os.path.join(output_folder, 'validation_data_full.csv'))
test_df = pd.read_csv(os.path.join(output_folder, 'test_data_full.csv'))

# Load the vectorizer and transform
vectorizer = joblib.load(os.path.join(output_folder, 'vectorizer.pkl'))

# Clean function
def clean_text_for_model(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
train_df['clean_text'] = train_df['text'].apply(clean_text_for_model)
val_df['clean_text'] = val_df['text'].apply(clean_text_for_model)
test_df['clean_text'] = test_df['text'].apply(clean_text_for_model)

# Remove empty texts
train_df = train_df[train_df['clean_text'].str.len() > 10]
val_df = val_df[val_df['clean_text'].str.len() > 10]
test_df = test_df[test_df['clean_text'].str.len() > 10]

# Transform to TF-IDF
X_train = vectorizer.transform(train_df['clean_text'])
X_val = vectorizer.transform(val_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

y_train = train_df['intent']
y_val = val_df['intent']
y_test = test_df['intent']

print(f"\nData shapes:")
print(f"  Training: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")

# DEFINE MODELS TO TEST

models = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'params': {
            'C': [0.1, 0.5, 1.0, 2.0],
            'solver': ['liblinear', 'lbfgs']
        }
    },
    'SVM (RBF)': {
        'model': SVC(random_state=42, class_weight='balanced', probability=True),
        'params': {
            'C': [0.1, 0.5, 1.0, 2.0],
            'gamma': ['scale', 'auto']
        }
    },
    'SVM (Linear)': {
        'model': SVC(kernel='linear', random_state=42, class_weight='balanced', probability=True),
        'params': {
            'C': [0.1, 0.5, 1.0, 2.0]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, 30],
            'min_samples_split': [2, 5, 10]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }
    },
    'K-Nearest Neighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance']
        }
    },
    'MLP Neural Network': {
        'model': MLPClassifier(random_state=42, max_iter=1000),
        'params': {
            'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
            'alpha': [0.0001, 0.001, 0.01],
            'learning_rate_init': [0.001, 0.01]
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'max_depth': [10, 20, 30, 50],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    }
}

# TRAIN AND EVALUATE EACH MODEL

print("\n" + "=" * 60)
print("TRAINING AND EVALUATING MODELS")
print("=" * 60)

results = []

for model_name, model_config in models.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    # Grid search for best parameters
    grid_search = GridSearchCV(
        model_config['model'],
        model_config['params'],
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    
    # Evaluate on validation set
    y_val_pred = best_model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    
    # Evaluate on test set
    y_test_pred = best_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
    
    print(f"\nBest parameters: {best_params}")
    print(f"Validation Accuracy: {val_accuracy:.3f}")
    print(f"Validation F1 (macro): {val_f1:.3f}")
    print(f"Test Accuracy: {test_accuracy:.3f}")
    print(f"Test F1 (macro): {test_f1:.3f}")
    
    # Save model
    joblib.dump(best_model, os.path.join(output_folder, f'{model_name.replace(" ", "_")}_model.pkl'))
    
    results.append({
        'Model': model_name,
        'Best Params': str(best_params),
        'Validation Accuracy': val_accuracy,
        'Validation F1': val_f1,
        'Test Accuracy': test_accuracy,
        'Test F1': test_f1
    })

# COMPARE ALL MODELS

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test Accuracy', ascending=False)

print("\nRanking by Test Accuracy:")
print(results_df[['Model', 'Test Accuracy', 'Test F1']].to_string(index=False))

# Save results
results_df.to_csv(os.path.join(output_folder, 'model_comparison_results.csv'), index=False)
print(f"\nResults saved to: {output_folder}\\model_comparison_results.csv")

# RECOMMENDATION

print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)

best_model_row = results_df.iloc[0]
print(f"\nBest performing model: {best_model_row['Model']}")
print(f"Test Accuracy: {best_model_row['Test Accuracy']:.3f}")

if best_model_row['Test Accuracy'] > 0.762:
    improvement = (best_model_row['Test Accuracy'] - 0.762) / 0.762 * 100
    print(f"Improvement over Logistic Regression: +{improvement:.1f}%")
else:
    print("Logistic Regression remains the best performer")

print(f"\nDetailed results saved to: {output_folder}\\model_comparison_results.csv")

COMPARING DIFFERENT ML MODELS ON BERT EMBEDDINGS

Data shapes:
  Training: (2233, 5000)
  Validation: (638, 5000)
  Test: (320, 5000)

TRAINING AND EVALUATING MODELS

Training Logistic Regression...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best parameters: {'C': 2.0, 'solver': 'lbfgs'}
Validation Accuracy: 0.729
Validation F1 (macro): 0.467
Test Accuracy: 0.750
Test F1 (macro): 0.486

Training SVM (RBF)...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best parameters: {'C': 2.0, 'gamma': 'scale'}
Validation Accuracy: 0.679
Validation F1 (macro): 0.304
Test Accuracy: 0.719
Test F1 (macro): 0.351

Training SVM (Linear)...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

Best parameters: {'C': 2.0}
Validation Accuracy: 0.735
Validation F1 (macro): 0.438
Test Accuracy: 0.750
Test F1 (macro): 0.423

Training Random Forest...
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best parameters: {'max_depth': 10, 'min_samples_split': 10,

KeyboardInterrupt: 